# Lab type: review
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: Anomaly Detection
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## Step 1: Generate Sensor Data

We simulate a manufacturing line with five sensor readings (temperature, pressure, vibration, humidity, voltage). Normal operation follows a multivariate Gaussian. A small number of injected anomalies represent fault conditions.

In [ ]:
# 300 normal operating points — correlated sensor readings
mean = np.array([70, 5.0, 0.3, 45, 230])
cov = np.array([
    [4.0, 1.2, 0.1, 0.0, 0.5],
    [1.2, 0.4, 0.0, 0.0, 0.1],
    [0.1, 0.0, 0.01, 0.0, 0.0],
    [0.0, 0.0, 0.0, 9.0, 0.0],
    [0.5, 0.1, 0.0, 0.0, 25.0]
])
X_normal = rng.multivariate_normal(mean, cov, size=300)

# 15 anomalous points — sensor combinations outside normal operating range
X_anomalies = np.array([
    [85, 7.5, 0.9, 55, 260],   # overheating + high vibration
    [60, 3.0, 0.6, 40, 210],   # under-pressure + elevated vibration
    [90, 8.2, 1.1, 60, 270],
    [55, 2.5, 0.7, 38, 205],
    [88, 7.8, 0.8, 50, 265],
    [92, 8.5, 1.2, 62, 275],
    [58, 2.8, 0.5, 35, 200],
    [86, 7.2, 0.9, 52, 255],
    [63, 3.5, 0.6, 42, 215],
    [91, 8.0, 1.0, 58, 268],
    [57, 2.6, 0.8, 37, 202],
    [87, 7.6, 1.1, 53, 262],
    [62, 3.2, 0.5, 41, 212],
    [89, 8.1, 0.9, 59, 272],
    [56, 2.7, 0.7, 36, 203],
])

# Combine and record ground-truth labels for later evaluation
X = np.vstack([X_normal, X_anomalies])
y_true = np.array([0] * 300 + [1] * 15)  # 0 = normal, 1 = anomaly

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset: {X.shape[0]} points, {X.shape[1]} features")
print(f"True anomaly rate: {y_true.mean()*100:.1f}%")

## Part 1: DBSCAN Noise Points

In [ ]:
# eps calibrated from the k-distance graph; min_samples chosen relative to dataset size
db = DBSCAN(eps=1.2, min_samples=8)
db_labels = db.fit_predict(X_scaled)

db_anomaly_pred = (db_labels == -1).astype(int)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = db_anomaly_pred.sum()

print(f"Clusters found: {n_clusters}")
print(f"Noise points (anomaly candidates): {n_noise} ({n_noise/len(X)*100:.1f}%)")
print()
print(classification_report(y_true, db_anomaly_pred, target_names=["normal", "anomaly"]))

**Question 1:** DBSCAN found one tight cluster and flagged noise points as anomaly candidates. If you tightened `eps` to get a better-defined cluster boundary, more sparse-but-normal points would also become noise. How would you decide, in practice, whether a noise point is a genuine anomaly or just a legitimate low-density observation?

*(Write your answer here.)*

**Question 2:** The precision and recall here are coupled to `eps` and `min_samples` — the same parameters that define your clusters. A colleague suggests running a grid search over `eps` to maximise recall for anomaly detection. What is wrong with this approach, and what would a better workflow look like?

*(Write your answer here.)*

## Part 2: Isolation Forest

In [ ]:
# Train on the full dataset; contamination set close to the true anomaly rate
iso = IsolationForest(contamination=0.05, random_state=42)
iso_preds = iso.fit_predict(X_scaled)  # 1 = inlier, -1 = anomaly
iso_scores = iso.decision_function(X_scaled)  # more negative = more anomalous

iso_anomaly_pred = (iso_preds == -1).astype(int)

print(f"Flagged as anomalies: {iso_anomaly_pred.sum()} ({iso_anomaly_pred.mean()*100:.1f}%)")
print()
print(classification_report(y_true, iso_anomaly_pred, target_names=["normal", "anomaly"]))

In [ ]:
# Score distribution: where normal and anomalous points sit on the decision boundary
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(iso_scores[y_true == 0], bins=30, alpha=0.6, label="Normal", color="steelblue")
ax.hist(iso_scores[y_true == 1], bins=10, alpha=0.7, label="Anomaly", color="tomato")
ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Decision boundary (score=0)")
ax.set_xlabel("Anomaly score (more negative → more anomalous)")
ax.set_ylabel("Count")
ax.set_title("Isolation Forest: Score Distributions")
ax.legend()
plt.tight_layout()
plt.show()

**Question 3:** `contamination=0.05` was set because the true anomaly rate is approximately 4.8%. In a real deployment you rarely know the true rate. The score distribution plot above shows how cleanly the two groups separate. Given what you see, what would happen to precision and recall if you changed `contamination` to `0.10`? When would doubling the contamination estimate be the right business decision?

*(Write your answer here.)*

**Question 4:** The code calls `fit_predict(X_scaled)` on the full dataset — normal and anomalous points together. The lesson notes that a train/test split is the correct approach for evaluation. In this specific lab, we already have ground-truth labels (`y_true`). Does fitting on the full dataset here introduce a valid concern? Under what real-world condition would it be a serious problem?

*(Write your answer here.)*

## Part 3: PCA Reconstruction Error

In [ ]:
# Split before fitting the PCA so the reconstruction model never sees test anomalies
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_true, test_size=0.3, random_state=42, stratify=y_true
)

In [ ]:
# 3 components retain the dominant variance of the 5-feature normal distribution
pca = PCA(n_components=3)
pca.fit(X_train)

print(f"Variance explained by 3 components: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Per-component: {pca.explained_variance_ratio_.round(3)}")

In [ ]:
# Reconstruction error: how much information is lost projecting to 3D and back
X_test_reconstructed = pca.inverse_transform(pca.transform(X_test))
reconstruction_error = np.mean((X_test - X_test_reconstructed) ** 2, axis=1)

# Threshold: flag the top 5% most poorly reconstructed points
threshold = np.percentile(reconstruction_error, 95)
pca_anomaly_pred = (reconstruction_error > threshold).astype(int)

print(f"Threshold (95th percentile): {threshold:.4f}")
print(f"Flagged as anomalies: {pca_anomaly_pred.sum()} ({pca_anomaly_pred.mean()*100:.1f}%)")
print()
print(classification_report(y_test, pca_anomaly_pred, target_names=["normal", "anomaly"]))

In [ ]:
# Reconstruction error distributions for normal vs anomalous test points
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(reconstruction_error[y_test == 0], bins=20, alpha=0.6, label="Normal", color="steelblue")
ax.hist(reconstruction_error[y_test == 1], bins=8, alpha=0.7, label="Anomaly", color="tomato")
ax.axvline(threshold, color="black", linestyle="--", linewidth=1, label=f"Threshold ({threshold:.3f})")
ax.set_xlabel("Mean squared reconstruction error")
ax.set_ylabel("Count")
ax.set_title("PCA: Reconstruction Error Distributions")
ax.legend()
plt.tight_layout()
plt.show()

**Question 5:** The threshold is set at the 95th percentile of reconstruction error on the test set. By construction, this flags roughly 5% of test points regardless of whether they are anomalies. Looking at the plot above: does this threshold appear well-calibrated for this dataset? What would you change, and why?

*(Write your answer here.)*

**Question 6:** PCA was fit on `X_train` only. Why does it matter that the PCA is fit exclusively on training data? Describe specifically what would go wrong if you instead fit PCA on `X_scaled` (the full dataset including test anomalies) before splitting.

*(Write your answer here.)*

## Part 4: Method Agreement

In [ ]:
# Re-run Isolation Forest on the same train/test split for a fair comparison
iso2 = IsolationForest(contamination=0.05, random_state=42)
iso2.fit(X_train)
iso2_preds = iso2.predict(X_test)
iso2_anomaly_pred = (iso2_preds == -1).astype(int)

# Agreement: both methods flag the same point
both_flag = (iso2_anomaly_pred == 1) & (pca_anomaly_pred == 1)
either_flag = (iso2_anomaly_pred == 1) | (pca_anomaly_pred == 1)

print(f"Flagged by Isolation Forest only: {((iso2_anomaly_pred==1) & (pca_anomaly_pred==0)).sum()}")
print(f"Flagged by PCA only:              {((iso2_anomaly_pred==0) & (pca_anomaly_pred==1)).sum()}")
print(f"Flagged by both (high confidence): {both_flag.sum()}")
print()
print("High-confidence anomalies (both methods agree):")
print(classification_report(y_test, both_flag.astype(int), target_names=["normal", "anomaly"]))

**Question 7:** Some test anomalies were flagged by only one method. Describe a plausible scenario — using the sensor data context — where PCA would flag a point that Isolation Forest misses. Then describe the reverse: a scenario where Isolation Forest flags a point that PCA misses.

*(Write your answer here.)*

**Question 8:** The "high confidence" set (both methods agree) has higher precision than either method alone. But requiring agreement also tends to lower recall. In the fraud detection context from the lesson — where missing a fraud is far more costly than a false alarm — would you use the agreement-only set, either method alone, or the union of both? Justify your answer in terms of the precision/recall tradeoff.

*(Write your answer here.)*

## Summary

> **Final check:** Answer in one sentence each.

1. What is the main risk of using DBSCAN noise points as anomaly detectors without further validation?
2. `contamination` in Isolation Forest and the percentile threshold in PCA reconstruction error serve analogous roles. What is the shared judgment they both require?
3. Why is method agreement (both methods flagging the same point) a useful proxy for confidence when no ground-truth labels exist?

*(Write your answers here.)*